# Steps 4 & 5 — Oracle Authorization & Household Registration

Replaces the Hardhat console for these two steps with `web3.py`, since the console loses state between commands and can be flaky on Sepolia.

This notebook keeps the `w3`, `account`, and contract objects alive in memory across cells, and uses explicit nonce handling + `wait_for_transaction_receipt` so you always know a tx actually landed before moving on — that's usually what makes the console feel unreliable (it doesn't wait, so you queue up txs with stale nonces).

**Assumption:** `config.json` has `oracle_storage_address`, `p2p_market_address`, and a `households` list of `{"address": "0x..."}`. Adjust the key names in the cell below if yours differ.

In [1]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from web3 import Web3

load_dotenv()  # reads python/.env

with open("config.json") as f:
    config = json.load(f)

config

{'comment': 'Konfiguration für Energy Trading Challenge Simulation',
 'simulation': {'slot_duration_seconds': 60,
  'comment_slot': '1 Slot = 1 reale Minute = 15 Minuten in der Simulation'},
 'households': [{'id': 'house_01',
   'address': '0x4022d2250AB1E76d3fcbCcf18d39656f4559b83c',
   'type': 'producer_consumer',
   'pv_peak_kwp': 8.0,
   'battery_capacity_kwh': 10.0,
   'battery_max_rate_kwh': 3.0,
   'base_consumption_kwh_per_hour': 0.4},
  {'id': 'house_02',
   'address': '0xF49153d700AD86CA224f7B2064F541278FE1c320',
   'type': 'producer_consumer',
   'pv_peak_kwp': 5.0,
   'battery_capacity_kwh': 7.5,
   'battery_max_rate_kwh': 2.5,
   'base_consumption_kwh_per_hour': 0.6}],
 'weather': {'comment': 'Tagesgang wird simuliert: Sonnenaufgang bis -untergang',
  'max_irradiance_wm2': 900,
  'base_temperature_c': 18.0,
  'cloud_cover_pattern': 'sinusoidal_with_noise'},
 'blockchain': {'rpc_url': 'https://ethereum-sepolia-rpc.publicnode.com',
  'chain_id': 11155111,
  'oracle_storage_a

## Connect

Set `RPC_URL` and `PRIVATE_KEY` in your `.env` (mirroring `.env.example`). Using an env var for the RPC endpoint (Infura/Alchemy Sepolia URL) rather than hardcoding it keeps secrets out of the notebook if you ever share it.

In [ ]:
w3 = Web3(Web3.HTTPProvider(os.environ["RPC_URL"]))
assert w3.is_connected(), "RPC connection failed — check RPC_URL"

account = w3.eth.account.from_key(os.environ["bb0a0d5667ab20e5f60c80e65afef6188f90a92cf72e689aca335b2bf9dc57b7"])
print("Connected, chain id:", w3.eth.chain_id)
print("Account:", account.address)
print("Balance (ETH):", w3.from_wei(w3.eth.get_balance(account.address), "ether"))

KeyError: 'RPC_URL'

## Load contracts

Reads the ABIs you already copied to `python/abi/` per the README setup step.

In [ ]:
def load_contract(abi_path: str, address: str):
    with open(abi_path) as f:
        artifact = json.load(f)
    abi = artifact["abi"] if "abi" in artifact else artifact
    return w3.eth.contract(address=Web3.to_checksum_address(address), abi=abi)

oracle_storage = load_contract("abi/OracleStorage.json", config["oracle_storage_address"])
p2p_market = load_contract("abi/P2PEnergyMarket.json", config["p2p_market_address"])

oracle_storage.address, p2p_market.address

## Reusable send-tx helper

This is the part that actually fixes the reliability problem: explicit nonce (fetched fresh each call, using `'pending'` so back-to-back sends don't collide), a fixed gas limit instead of estimation (estimation is where public-testnet flakiness usually bites), and a blocking wait for the receipt so you get an immediate pass/fail instead of silent queued state.

Tune `gas` if your calls run out of gas — check the receipt's `gasUsed` on a successful run and pad it a bit.

In [ ]:
def send_tx(contract_function, gas: int = 200_000):
    nonce = w3.eth.get_transaction_count(account.address, "pending")
    tx = contract_function.build_transaction({
        "from": account.address,
        "nonce": nonce,
        "gas": gas,
        "chainId": w3.eth.chain_id,
    })
    signed = account.sign_transaction(tx)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    status = "OK" if receipt.status == 1 else "REVERTED"
    print(f"{status} | tx {tx_hash.hex()} | gasUsed {receipt.gasUsed}")
    return receipt

## Step 4 — Authorize the oracle wallet

Worked example. If `account` above *is* the deployer, this is technically redundant (deployer is oracle by default) — but useful if you're authorizing a separate oracle-writer wallet, e.g. the one `oracle_writer.py` uses.

In [ ]:
oracle_wallet_address = "0xORACLE_WALLET_ADDR"  # the wallet oracle_writer.py signs with

send_tx(oracle_storage.functions.authorizeOracle(Web3.to_checksum_address(oracle_wallet_address)))

## Step 5 — Register households on P2PEnergyMarket

Note: Oracle-side household registration (`registerHousehold` on `OracleStorage`) already happens automatically when you start `oracle_writer.py` — per the README, you don't need to do that here. What's *not* automatic is registering each household on `p2p_market` itself.

Your turn: loop over `config["households"]` and call `p2p_market.functions.registerHousehold(...)` via `send_tx` for each address. A couple of things worth thinking through as you write it:
- Do you want to `send_tx` sequentially (safe, slower) or fire all txs then wait (faster, but you'd need to manage nonces yourself instead of relying on `'pending'` inside the helper)? For a handful of households, sequential is simpler and avoids nonce race conditions entirely.
- Consider what happens if a household is already registered — does the contract revert, or is it idempotent? Worth checking the contract source so a re-run of this cell doesn't blow up on already-registered addresses.

In [ ]:
# TODO: loop config["households"] and send_tx(p2p_market.functions.registerHousehold(...))
for household in config["households"]:
    pass